In [61]:
# Initialize Otter
import otter
grader = otter.Notebook("hw6.ipynb")

# FILL IN YOUR NAME AND THE NAME OF YOUR PEER (IF ANY) BELOW

**Name**: Pei Ma

**Peer**: ChatGPT

## Collaboration policy
Students are responsible for writing their own quizzes, assignments, projects, and exams. For homework assignments, students are welcome (and encouraged) to discuss problems with one peer, **but each student must write their own assignment writeup and code individually**. The peer must be listed at the top of the writeup for each assignment. *Note: Homework assignments are the primary learning mechanism for this course—you learn by doing. I strongly discourage the use of AI assistants in solving assignments. That said, I will treat AI assistants as peers: you are allowed to discuss problems with an AI assistant, but it is considered cheating to directly obtain an answer by querying the assistant. Please credit any AI assistant that you use.*

In [62]:
# Run this cell before starting
import numpy as np

# Homework 6 -- CNNs (100 pts)

**Due:** Wednesday, October 21st, 2026 at 11:59 pm

This homework builds on the material in the slides and the MIT course notes on CNNs.

We will use Jupyter/Colab notebooks throughout the semester for writing code and generating assignment outputs. 

Note that some questions have public tests, and others do not. These are only intended to provide insight into what may not be working in your solution, but they generally do not count toward your grade. Passing all public tests does not guarantee that you will obtain full marks for a problem. The per-question score given to you by Gradescope is the only feedback you will obtain about your grade. It is possible that `print` statements may cause your code to receive no marks for a given question, so we encourage you to remove/comment out any `print` statements before submitting your assignment. **If you pass public tests but fail to obtain full marks on Gradescope, it is your responsibility to debug your code further (e.g., by devising additional test cases).**

## 1) CNN Architectures (10 pts)

In order to get practice with the structural choices in a CNN and understand how parameters are shared, we will consider how many separate weights are in NNs of different architectures.

**Hint:** For all the questions in this section, if you get stuck, try drawing a picture/diagram of what the network would look like.


### 1.1)

In a fully-connected feedforward network, the number of weights (including biases) for one layer with 100 inputs and 80 outputs is:

_Points:_ 2

In [63]:
n_inputs = 100 
n_outputs = 80
ans_11 = (n_inputs + 1) * n_outputs

In [64]:
grader.check("q1.1")

q1.1 results: All test cases passed!

### 1.2)
Consider a CNN for 1D data. Suppose a convolutional layer receives an input of size 100. This layer has 10 filters, each of size 5. Suppose we use a zero-padding of size 2 on both ends of the input, and a stride of 1.

What are the dimensions of the output of this convolutional layer? Hint: have a look at the MIT course notes for how to calculate this.

Provide a list of numbers of appropriate length.

_Points:_ 2

In [65]:
ans_12 = [100, 10]

In [66]:
grader.check("q1.2")

q1.2 results: All test cases passed!

### 1.3)

How many weights (including bias) are needed to specify the map from the input to the output of this convolutional layer?

_Points:_ 2

In [67]:
ans_13 = 10 * 5 + 10

In [68]:
grader.check("q1.3")

q1.3 results: All test cases passed!

### 1.4)
Now consider a zero-padded max pooling layer with an input of size 100, one pooling filter of size 3, and a stride of 2. Assume zero-padding of size 1.

What are the dimensions of the output of this pooling layer?

Provide a list of numbers of appropriate length. 

_Points:_ 2

In [69]:
# only one pooling filter, so it should be 1 dim 
ans_14 = [50]

In [70]:
grader.check("q1.4")

q1.4 results: All test cases passed!

### 1.5)

How many weights (including bias) are needed to specify the max pooling layer?

_Points:_ 2

In [71]:
ans_15 = 0

## 2) Conversion from Convolutional to Fully Connected (5 pts)

Let's now formalize the relationship between convolutional layers and fully connected layers.

Suppose we have:
- a 1D input $x$ of size 4
- a filter $f$ of size 3: $(w_1, w_2, w_3)$ with stride of 1 and no bias.
- a 1D output $z$ of size 4, produced by convolving $f$ with $x$. To preserve size, we use padding of size 1 on both sides.

Assume that when doing convolutions, inputs that fall outside the image bounds are treated as zeros.

**The effect of convolving this filter with the input layer $x$ is actually equivalent to constructing a matrix $W$ of weights between two layers of a fully connected network**, i.e.:
$$z=W^\top x\enspace,$$
where $W$ is a 4-by-4 matrix (and there is no bias term). Note though that in the conversion, the same weight may occur more than once in the matrix $W$.

Which of the following weight matrices corresponds to $W^\top$ in the equation above? That is, $z$ was produced using a convolution between $f$ and $x$, but if we want to produce $z$ using $W$ and $x$, as defined above, what would $W^\top$ be?

Hint: What is the output $z$, in terms of the elements of $w$ and $x$?

a. $\left[ \begin{array}{c} w_2 & w_3 & 0 & 0\\w_1 & w_2 & w_3 & 0\\ 0 & w_1 & w_2 & w_3\\0 & 0 & w_1 & w_2\\ \end{array} \right]$

b. $\left[ \begin{array}{c} w_2 & w_1 & 0 & 0\\w_3 & w_2 & w_1 & 0\\ 0 & w_3 & w_2 & w_1\\0 & 0 & w_3 & w_2\\ \end{array} \right]$

c. $\left[ \begin{array}{c} w_1 & 0 & 0 & 0\\w_2 & w_1 & 0 & 0\\ w_3 & w_2 & w_1 & 0\\0 & w_3 & w_2 & w_1\\ \end{array} \right]$

d. $\left[ \begin{array}{c} w_1 & w_2 & w_3 & 0\\ 0 & w_1 & w_2 & w_3\\0 & 0 & w_1 & w_2\\0 & 0 & 0 & w_1\\ \end{array} \right]$



_Points:_ 5

In [72]:
# set ans = "a" for (a), ans = "b" for (b), ...
ans_2 = "a"

## 3) Convoluted Network (15 pts)

We will explore how convolutional neural networks operate by designing one. Our objective is to be able to locate the pattern

<img src="https://jorge-a-mendez.github.io/files/ese577f25/hw6/convoluted1.png" width=20%>

in an image. Throughout this problem, treat dark squares as having value $+1$ and light squares as having value $-1$.

### 3.1)

Consider the image that would result from convolving the image below with a filter that is the same as the pattern above. (Use our definition of convolution, in which we slide the **center of** the filter over the image and compute the dot product.) Assume that the edges are padded with $-1$ and that we use a stride of $1$.

<img src="https://jorge-a-mendez.github.io/files/ese577f25/hw6/convoluted2.png" width=30%>

Which pixel in the **resulting** image has the largest value, and what is the resulting value?

Provide your answer as a list `[[row, column], value]`, where the first entry is the pixel indices and the second is its value. The pixels are indexed such that the top-left is `[0, 0]`, and the first index refers to the row.


_Points:_ 5

In [73]:
ans_31 = [[1, 3], 9]

In [74]:
grader.check("q3.1")

q3.1 results: All test cases passed!

### 3.2)

In order to detect this pattern, we could create a very simple network that has:
- a convolutional layer with a single filter, corresponding to the desired pattern, applied with a stride of 1,
- no activation function,
- a max-pooling layer with input size equal to the image size, and finally
- a single ReLU unit

Note that there is no need for additional `flatten` or fully-connected layers because a whole image (result of convolving with our filter) goes into the max pooling and a single value comes out.

Provide a value for the offset $W_0$ in the input to the ReLU that, for any image, would guarantee the output of the ReLU is positive if an only if there is an exact match for this pattern in the image.


_Points:_ 5

In [75]:
W0_32 = -8

In [76]:
grader.check("q3.2")

q3.2 results: All test cases passed!

### 3.3)

Kan Volution thinks that instead of having this single convolution layer with a single filter matching the whole desired pattern, it would be better to start with a convolutional layer with four smaller filters, shown below (with black squares having a value of $+1/4$ and white squares having a value of $-1/4$):

<img src="https://jorge-a-mendez.github.io/files/ese577f25/hw6/convoluted3.png" width=50%>

It is slightly unusual to have $2\times2$ filters (usually they have odd dimension). When we apply them, we change the convention a little, and place the upper-left pixel (instead of the usual center) of the filter on top of the image pixel whose value we are computing. We use $-1$ padding for one column on the right and one row on the bottom of the input image.

The following images are the result of convolving the original image with these 4 simple filters and running through a ReLU.

Black squares have a value $+1$, grey squares have a value $+0.5$, and the rest have value $0$. Try to convince yourself that these 4 resulting images (and their values) make sense.

<img src="https://jorge-a-mendez.github.io/files/ese577f25/hw6/convoluted4.png" width=70%>

Now, the next layer of Kan's network takes in this 3D tensor (of shape $5\times5\times4$, with $4$ channels) and applies a single $2\times2\times4$ filter.

Specify a filter on this next layer that will generate a final image with a high value at the pixel located at the upper left corner of the original 3x3 pattern we are trying to detect, and lower values elsewhere.

Provide your answer as a (nested) list of the four filters. For example `[[[-1, 1], [1, 1]], [[1, -1], [1, 1]], [[1, 1], [-1, 1]], [[1, 1], [1, -1]]]` describes four 2-by-2 filters here each filter has one light pixel. In particular, the first filter `[[-1, 1], [1, 1]]` has a light pixel at the top-left; the second filter at top-right; the third at bottom-left; and the fourth at bottom-right.

Your list should follow the same shape/location pattern as the example, and each of the weights in your filters should be either $1$ or $-1$.

_Points:_ 5

In [77]:
filters_33 = [[[-1, -1],[-1, 1]], [[-1, -1], [1, -1]], [[1, -1], [-1, -1]], [[-1, 1], [-1, -1]]]

In [78]:
grader.check("q3.3")

q3.3 results: All test cases passed!

## 4) Kat's Cat (20 pts)

### 4.1)

In this question, we will train CNNs to classify images. As a warmup, we first look at a CNN with one convolutional layer and one activation layer with a one-dimensional input.
- Assume there is a sign activation function: $\mathrm{sign}(z) = \begin{cases}1 \quad\text{if\ } z>0\\0 \quad\text{if\ } z<0 \end{cases}$. For this problem , **we will not allow boundary cases, so an error will be raised if $z=0$ is the input to any $\mathrm{sign}(z)$**
- Assume we will use zero-padding of size 1 to handle the image boundary
- Assume a stride of 1

#### 4.1.1)
Provide a 3 x 1 filter and offset that produces the output below when applied to the corresponding input:
```
input:   [0 0 1 1 0 0 1 0]
output:  [0 0 0 0 0 0 1 0]
```

Enter the weights of a 3 x 1 filter and the bias as a list: $[w_1, w_2, w_3, b]$.


_Points:_ 5

In [79]:
ans_411 = [-2, 2, -2, -1]

In [80]:
grader.check("q4.1.1")

q4.1.1 results: All test cases passed!

#### 4.1.2)

Now imagine that we take two images:
```
[0 0 0 0 0 0 1 0]
[0 0 0 0 0 1 0 0]
```
and treat them as two channels that are the outputs of a convolutional layer. We convolve them with a (1 x 1 x 2) tensor filter `[[[1, 1]]]` to obtain a 1-channel 1D image. What is the resulting image?

Enter the resulting image as a list of the appropriate length.


_Points:_ 5

In [81]:
ans_412 = [0, 0, 0, 0, 0, 1, 1, 0]

In [82]:
grader.check("q4.1.2")

q4.1.2 results: All test cases passed!

### 4.2)

Kat trains a CNN classifier to recognize their cat, but the cat is always in the lower left quadrant of the 100 x 100 training images, and it usually takes up a 20 x 20 image patch and there is no sub-image of size less than 20 x 20 that will reliably recognize the cat.

For the rest of this problem, we will be talking about PyTorch. Please view this note on PyTorch for more information about how to use it:

# A note on PyTorch

In HW5, you implemented many building blocks of neural networks (like **Linear**, **Tanh**, **ReLU**, **SoftMax**, **NLLM**) from scratch. When implementing neural networks in practice, we use a machine learning framework that already has these building blocks implemented instead of building everything from scratch. In this homework, we'll use PyTorch, a Python framework for implementing neural networks, to construct autoencoders. PyTorch is currently one of the most popular frameworks within the machine learning community, so it's worth becoming more familiar with it.

In particular, PyTorch uses the **Tensor** data type to encode all the objects it works on.  In our usage, a tensor is the generalization of the idea of an array (which is typically 2D) to a structure that can have arbitrarily many dimensions.   We will use tensors to represent data sets, batches of data, images, batches of images, weights, activations, etc. They are very versatile but can be a little bit difficult to conceptualize especially when you're starting out.  You'll get the hang of it! We recommend looking at this very good [primer on PyTorch tensors](https://pytorch.org/tutorials/beginner/blitz/tensor_tutorial.html).

PyTorch creates networks in a modular fashion, by chaining different [torch.nn.Modules](https://pytorch.org/docs/stable/nn.html) together. Some helpful modules for this assignment include:
- [nn.Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) -- implements (the pre-activations for) a fully-connected layer, what we call $W^\top x + b$.
- [nn.Conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) -- implements a convolutional layer.
- [nn.MaxPool2d](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) -- implements a MaxPooling layer.
- [nn.Upsample](https://pytorch.org/docs/stable/generated/torch.nn.Upsample.html) -- implements an Upsampling layer. You can think of this as the opposite operation to a MaxPooling layer.
- [nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html) -- implements ReLU activation.
- [nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html) -- implements element-wise Mean Squared Error.
- [nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) -- implements Categorical Cross-Entropy (**Softmax + NLLM**).
- [nn.Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html) -- implements reshaping of matrices/tensors into vectors.
- [nn.Dropout](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html) -- implements random dropout of input neurons, preventing overfitting.
- Because a linear activation function just returns the same value as its input, we can implement a linear activation by just omitting the activation function.

For an example of how all these pieces fit together to create a network, take a look at this [quickstart tutorial](https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html) from the PyTorch website.

#### 4.2.1)

Kat makes a CNN in Pytorch with layers
```
[
    nn.Conv2d(1, 1, kernel_size = (20, 20), stride = 1, padding = 10),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size = (100, 100)),
    nn.Flatten(),
    nn.Linear(1, 1)
]
```
and `CrossEntropyLoss`. It has 100% training accuracy. Statement: it will it have high accuracy on test images with the cat in a 20-pixel window in the upper right quadrant. True or false?


_Points:_ 2

In [83]:
# set ans = True if the statement above is true, or ans = False otherwise

# a 20x20 filter would reliably recognize the cat, according to the problem description. 
# so, the 20x20 filter in Conv2d should be able to achieve h igh accuracy on test images 
# whether the cat is in the bottom left quadrant or upper right should not matter for the filter

ans_421 = True

#### 4.2.2)

Assume Kat's CNN were instead
```
[
    nn.Conv2d(1, 10, kernel_size = (3, 3), stride = 1, padding = 1),
    nn.ReLU(),
    nn.Flatten(),
    nn.Linear(100000, 1)
]
```

Statement: it should be possible to train it to reliably recognize the cat in the training data. True or false?|

_Points:_ 2

In [84]:
# set ans = True if the statement above is true, or ans = False otherwise

# this method should combine info from many 3x3 regions across the image to recognize the entire cat

ans_422 = False

#### 4.2.3)

Now assume Kat's CNN were, instead
```
[
    nn.Conv2d(1, 10, kernel_size = (3, 3), stride = 1, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size = (3, 3), stride = 3),
    nn.Conv2d(10, 10, kernel_size = (3, 3), stride = 1, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size = (3, 3), stride = 3),
    nn.Conv2d(10, 1, kernel_size = (3, 3), stride = 1, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size = (3, 3), stride = 3),
    nn.Flatten(),
    nn.Linear(9, 1)
]
```
Statement: it should be possible to train it to reliably recognize the cat in the training data. True or false?

_Points:_ 2

In [85]:
# set ans = True if the statement above is true, or ans = False otherwise

# convolution with 3x3 filter to 10 x (100x100) output
# ReLU 
# MaxPool: takes output, and take max of each 3x3 window, to get 10x(33x33)

# then convolution with 3x3 filter again  
# ReLU
# MaxPool: takes output and takes max of 3x3 window, 10x(11x11)

# conv with 10x(3x3) to get 1x(11x11)
# ReLU
# MaxPool: -> 1x(3x3)

#flatten to get 9 values 
#linear to get a scalar score 

ans_423 = True

#### 4.2.4)
Kat's housemate Kit thinks a fully-connected network with many layers would be a better idea. So Kit makes each $100\times100$ image into a single input vector of size $10,\!000$ and runs it through a standard fully-connected network with 3 hidden layers, with non-linear activations and a single output. Statement: it would be possible for Kit to train their network to have high training accuracy on Kat's training data. True or false?

_Points:_ 2

In [86]:
# set ans = True if the statement above is true, or ans = False otherwise
ans_424 = True

#### 4.2.5)

Statement: Kit's network would have high accuracy on test data with cats in the upper right quadrant. True or false?

_Points:_ 2

In [87]:
# set ans = True if the statement above is true, or ans = False otherwise
ans_425 = False

## 5) Image Classification (50 pts)

For the rest of this assignment, we recommend that you move over to running code on Google Colab, using a GPU. You should make sure your Notebook is saved on Google Drive, so that your changes are saved (**otherwise you will lose your progress!**). This assignment will be evaluated a bit differently from previous ones: instead of you entering code for me to run and evaluate, you will instead be modifying the code at the bottom of this notebook to obtain your answers, which you will submit in the following questions.

We'll be working with the MNIST handwritten digit image dataset. This is a collection of 70,000 black and white images of handwritten digits (although the actual pixel values are grayscale, not exactly black or white). The original dataset has integer pixel values from `[0, 255]`, but we will scale our pixel values to `[0., 1.]` floats. Each image is (28 x 28) pixels. the labels are integers 0-9, so our output layer will always have 10 units. Because we are doing classification in this problem, we will use softmax activation on the output layer and negative log-likelihood loss.

<img src="https://jorge-a-mendez.github.io/files/ese577f25/hw6/mnist_dataset_examples.jpeg" width=60%>

*Examples of handwritten digits from the MNIST dataset*

### 5.1) No hidden layers

With one epoch of training, what is the validation accuracy of a network **with no hidden units** on this data?

Enter a number between 0 and 1 (to three decimal places):


_Points:_ 10

In [88]:
ans_51 = 0.852

In [89]:
grader.check("q5.1")

q5.1 results: All test cases passed!

### 5.2) One fully-connected hidden layer

Using one epoch of training, try a single hidden layer, followed by a ReLU activation layer before the final output layer, and gradually increase the units; specifically, try (128, 256, 512, 1024) units and observe the results. What are the accuracies on the validation set?

Enter a Python list of 4 numbers between 0 and 1 (to three decimal places):

_Points:_ 10

In [90]:
ans_52 =  [0.898, 0.908, 0.920, 0.930]

In [91]:
grader.check("q5.2")

q5.2 results: All test cases passed!

### 5.3) Two fully-connected hidden layers

Now, try a network with two hidden layers (for 1 epoch):
- A fully-connected layer with 512 hidden units
- A ReLU activation layer
- A fully-connected layer with 256 hidden units
- A ReLU activation layer

What is the accuracy on the test set? 

Enter a number between 0 and 1 (to three decimal places):

_Points:_ 5

In [92]:
ans_531 = 0.966

In [93]:
grader.check("q5.3.1")

q5.3.1 results: All test cases passed!

Now run the model for 20 epochs. What is the validation accuracy after the 1st, 5th, 10th, and 15th epochs?

Enter a Python list of 4 numbers between 0 and 1 (to three decimal places):

_Points:_ 10

In [94]:
ans_532 = [0.926, 0.962, 0.971, 0.975]

In [95]:
grader.check("q5.3.2")

q5.3.2 results: All test cases passed!

### 5.4) Convolutional architectures

Finally, build a convolutional network with the following structure:
- A convolutional layer (without padding) with 32 filters of size 3 x 3
- A ReLU activation layer
- A max pooling layer with size 2 x 2
- A convolutional layer (without padding) with 64 filters of size 3 x 3
- A ReLU activation layer
- A max pooling layer with size 2 x 2
- A flatten layer (will flatten to size 1600 = 5 x 4 x 64; try to figure out why!)
- A fully-connected layer with 128 units
- A ReLU activation layer
- A dropout layer with drop probability 0.5
- A fully connected laer with 10 output units

To define Convolutional and max pooling layers in PyTorch, use the following syntax:
```
c = Conv2d(in_channels=i, out_channels=o, kernel_size=filter_size)
m = MaxPool2d(kernel_size=filter_size)
```
where `i` and `o` are integers and `filter_size` can either be an integer (for square filters) or a tuple (for non-square filters, e.g. `(2, 3)` for a 2x3 filter).

For all convolutional layers, you can assume that there is no padding and that stride=1.

Train the model for one epoch and report your test accuracy.

Enter a number between 0 and 1 (to three decimal places):


_Points:_ 12

In [96]:
ans_54 = 0.983

In [97]:
grader.check("q5.4")

q5.4 results: All test cases passed!

Now, assume that the accuracy of getting one digit correct is 0.95. Assume that the predictions of digits are independent from each other. What is the probability of reading a full 5-digit zip code correctly?

Enter a number between 0 and 1 (to **four decimal places**):

_Points:_ 3

In [98]:
ans_541 = 0.95**5

In [99]:
grader.check("q5.4.1")

q5.4.1 results: All test cases passed!

# CNN code

**If you haven't already, please hit :**

`File` -> `Save a Copy in Drive`

**to copy this notebook to your Google drive, and work on a copy. If you don't do this, your changes won't be saved!**

# **RUN THIS NOTEBOOK WITH GPU**

OTHERWISE YOU WILL SPEND A LONG TIME ON TRAINING

**To do this, go to `Runtime` > `Change runtime type` and under `hardware acceleration` set the dropdown to `GPU`.**

## Boilerplate

Run the following cells to load the data. No need to edit anything here.

In [100]:
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms

# To get the MNIST (digit images) dataset
from keras.datasets import mnist

torch.manual_seed(0)

In [101]:
# Download MNIST Data
(mnist_train, labels_train), (mnist_test, labels_test) = mnist.load_data()

# Load data as Numpy arrays of size (#datapoints, 28*28=784)
mnist_train = mnist_train.astype('float32') / 255.
mnist_test = mnist_test.astype('float32') / 255.
mnist_train = mnist_train.reshape((len(mnist_train), np.prod(mnist_train.shape[1:])))
mnist_test = mnist_test.reshape((len(mnist_test), np.prod(mnist_test.shape[1:])))

# Split test data into a test and validation set:
val_data = mnist_test[:(mnist_test.shape[0]//2),:]
test_data = mnist_test[(mnist_test.shape[0]//2):,:]
train_data = mnist_train

val_labels = labels_test[:(mnist_test.shape[0]//2)]
test_labels = labels_test[(mnist_test.shape[0]//2):]
train_labels = labels_train

# Display dataset information
print("Downloaded the following data:")
print(f"train_data has shape {train_data.shape}, containing {train_data.shape[0]} images represented as ({train_data.shape[1]}, 1) vectors")
print(f"val_data has shape {val_data.shape}, containing {val_data.shape[0]} images represented as ({val_data.shape[1]}, 1) vectors")
print(f"train_data has shape {test_data.shape}, containing {test_data.shape[0]} images represented as ({test_data.shape[1]}, 1) vectors")

Downloaded the following data:
train_data has shape (60000, 784), containing 60000 images represented as (784, 1) vectors
val_data has shape (5000, 784), containing 5000 images represented as (784, 1) vectors
train_data has shape (5000, 784), containing 5000 images represented as (784, 1) vectors


In [102]:
# We'll create a Dataset class to use with PyTorch's Built-In Dataloaders
class MNISTDataset(Dataset):
    '''
    A custom dataset class to use with PyTorch's built-in dataloaders.
    This will make feeding images to our models much easier downstream.

    data: np.arrays downloaded from Keras' databases
    vectorize: if True, outputed image data will be (784,)
                   if False, outputed image data will be (28,28)
    '''
    def __init__(self, data, labels, vectorize=True):
        self.data = data
        self.labels = labels
        self.vectorize = vectorize

    def __getitem__(self, idx):
        image_data = self.data[idx, :]
        image_data = image_data.reshape((1,28,28))
        if self.vectorize:
            image_data = image_data.reshape((784,))
        image_label = self.labels[idx]
        return image_data, image_label

    def __len__(self):
        return self.data.shape[0]

# Create MNISTDataset objects for each of our train/val/test sets
train_dataset = MNISTDataset(train_data, train_labels)
val_dataset = MNISTDataset(val_data, val_labels)
test_dataset = MNISTDataset(test_data, test_labels)

# Create a PyTorch dataloader for each train/val/test set
# We'll use a batch size of 256 for the rest of this assignment.
train_loader = DataLoader(train_dataset, batch_size=256)
val_loader = DataLoader(val_dataset, batch_size=256)
test_loader = DataLoader(test_dataset, batch_size=256)

# Create another dataset that doesn't reshape data
cnn_train_dataset = MNISTDataset(train_data, train_labels, vectorize=False)
cnn_val_dataset = MNISTDataset(val_data, val_labels, vectorize=False)
cnn_test_dataset = MNISTDataset(test_data, test_labels, vectorize=False)

# create new dataloaders without data reshaping
cnn_train_loader = DataLoader(cnn_train_dataset, batch_size=256)
cnn_val_loader = DataLoader(cnn_val_dataset, batch_size=256)
cnn_test_loader = DataLoader(cnn_test_dataset, batch_size=256)

# Display dataloader info
print("Created the following Dataloaders:")
print(f"train_loader has {len(train_loader)} batches of training data")
print(f"val_loader has {len(val_loader)} batches of validation data")
print(f"test_loader has {len(test_loader)} batches of testing data")

Created the following Dataloaders:
train_loader has 235 batches of training data
val_loader has 20 batches of validation data
test_loader has 20 batches of testing data


# Image Classification

### Train/test scripts

We'll first implement the functions to `train` and `test` our classifiers. We've written most of the functions for you. Your job is to fill in the calculation of the loss.

As a reminder: For our loss function we will be using categorical cross-entropy loss between the output of our  model and our ground truth labels.

Note that:
* Use a built-in module instead of implementing the loss yourself!
* We are expecting a one-line solution in each case. Don't overthink this!
* The line you fill in for both `train()` and `test()` should be the same.

In [103]:
# EDIT ME!

def train(model, device, train_loader, optimizer, val_loader=None, pbar=False):
    '''
    Function for training our networks. One call to train() performs a single
    epoch for training.

    model: an instance of our model, in this assignment, this will be your autoencoder

    device: either "cpu" or "cuda", depending on if you're running with GPU support

    train_loader: the dataloader for the training set

    optimizer: optimizer used for training (the optimizer implements SGD)

    val_loader: (optional) validation set to include
    '''

    # Set the model to training mode.
    model.train()

    #we'll keep adding the loss of each batch to total_loss, so we can calculate
    #the average loss at the end of the epoch.
    total_loss = 0
    total_correct = 0
    total_items = 0

    # We'll iterate through each batch. One call of train() trains for 1 epoch.
    # batch_idx: an integer representing which batch number we're on
    # input: a pytorch tensor representing a batch of input images.
    if pbar:
      train_loader = tqdm(train_loader)

    for batch_idx, (input,target) in enumerate(train_loader):
        # This line sends data to GPU if you're using a GPU
        input = input.to(device)
        target = target.type(torch.LongTensor).to(device)

        # Zero out gradients from previous iteration
        optimizer.zero_grad()

        # feed our input through the network
        output = model.forward(input)

        ## TODO: YOUR CODE HERE

        loss_function = nn.CrossEntropyLoss()
        loss_value = loss_function(output, target)

        ## END YOUR CODE

        # Perform backprop
        loss_value.backward()
        optimizer.step()

        #accumulate loss to later calculate the average
        total_loss += loss_value
        total_correct += torch.sum( torch.argmax(output, dim=1)==target )
        total_items += input.shape[0]


    return total_loss.item()/len(train_loader), (total_correct/total_items).item()

In [104]:
# EDIT ME!
def test(model, device, test_loader, pbar=False):
    '''
    Function for testing our models. One call to test() runs through every
    datapoint in our dataset once.

    model: an instance of our model, in this assignment, this will be your autoencoder

    device: either "cpu" or "cuda:0", depending on if you're running with GPU support

    test_loader: the dataloader for the data to run the model on
    '''
    # set model to evaluation mode
    model.eval()

    # we'll keep track of total loss to calculate the average later
    test_loss = 0
    total_correct = 0
    total_items = 0

    # don’t track gradients in testing, since no backprop
    with torch.no_grad():
        # iterate thorugh each test image
        if pbar:
          test_loader = tqdm(test_loader)

        for input, target in test_loader:

            # send input image to GPU if using GPU
            input = input.to(device)
            target = target.type(torch.LongTensor).to(device)

            # run input through our model
            output = model(input)

            ## TODO: YOUR CODE HERE

            loss_function = nn.CrossEntropyLoss()
            loss_value = loss_function(output, target)

            ## END YOUR CODE

            # Accumulate for accuracy
            test_loss += loss_value
            total_correct += torch.sum( torch.argmax(output, dim=1)==target )
            total_items += input.shape[0]

    # calculate average loss/accuracy per batch
    test_loss /= len(test_loader)
    accuracy = total_correct / total_items

    return test_loss.item(), accuracy.item()

### Fully Connected Classifiers

The code box below will be modified to use various fully-connected architectures for the following questions.

You will need to design your own `layers`, for example `layers=[nn.Linear(in_features=64, out_features=4)]` defines a single layer with 64 inputs and 4 output units.

In [105]:
# EDIT ME!

# define the layers of our model
layers = [
    ## TODO: YOUR CODE HERE
    # 5.1: no hidden units 
    # nn.Linear(784, 10)

    # 5.2: first value 
    # nn.Linear(784, 128), 
    # nn.ReLU(), 
    # nn.Linear(128, 10)

    # 5.2: second value 
    # nn.Linear(784, 256), 
    # nn.ReLU(), 
    # nn.Linear(256, 10)
    
    # 5.2: third value 
    # nn.Linear(784, 512), 
    # nn.ReLU(), 
    # nn.Linear(512, 10)
    
    # 5.2: fourth value 
    # nn.Linear(784, 1024), 
    # nn.ReLU(), 
    # nn.Linear(1024, 10)

    # 5.3 
    nn.Linear(784, 512),
    nn.ReLU(),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
]

# set number of epochs to train for
epochs = 1 ## Change me in problem 5.3!

# for 5.3.2
# epochs = 20 ## Change me in problem 5.3!

# check if running on CPU or GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# initialize a fully-connected classifier
fc_model = nn.Sequential(*layers).to(device)

# initialize our optimizer. We'll use Adam
optimizer = torch.optim.Adam(fc_model.parameters())

# train your classifier
for epoch in range(1, epochs+1):
    train_loss, train_acc = train(fc_model, device, train_loader, optimizer)
    val_loss, val_acc = test(fc_model, device, val_loader)
    print('Train Epoch: {:02d} \tTraining Loss: {:.6f} \tTraining Acc: {:.6f}\n \t\t\tValidation Loss: {:.6f} \tValidation Acc: {:.6f}\n'.format(epoch, train_loss, train_acc, val_loss, val_acc))

# test your model
test_loss, test_acc = test(fc_model, device, test_loader)
print('\n\t\t\tTest Loss: {:.6f} \t Test Accuracy: {:.6f}'.format(test_loss, test_acc) )

Train Epoch: 01 	Training Loss: 0.392109 	Training Acc: 0.890717
 			Validation Loss: 0.269863 	Validation Acc: 0.916200


			Test Loss: 0.138368 	 Test Accuracy: 0.961400


### Convolutional Classifier

The code box below will be modified to use for convolutional neural network architectures. Note that the code is slightly different, using a different dataloader that doesn't flatten/vectorize the images.

_Points:_ 0

In [106]:
# EDIT ME!

# check if running on CPU or GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# define the layers of our model
layers = [
    ## TODO: YOUR CODE HERE
    # 5.4
    nn.Conv2d(1, 32, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),

    nn.Conv2d(32, 64, kernel_size=3),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),

    nn.Flatten(),
    nn.Linear(1600, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 10)
]

# initialize a CNN classifier
cnn_model = nn.Sequential(*layers).to(device)

# initialize our optimizer. We'll use Adam
optimizer = torch.optim.Adam(cnn_model.parameters())

# set number of epochs to train for
epochs = 1

# train your classifier
for epoch in range(1, epochs+1):
    train_loss, train_acc = train(cnn_model, device, cnn_train_loader, optimizer)
    val_loss, val_acc = test(cnn_model, device, cnn_val_loader)
    print('Train Epoch: {:02d} \tTraining Loss: {:.6f} \tTraining Acc: {:.6f} \tValidation Loss: {:.6f} \tValidation Acc: {:.6f}'.format(epoch, train_loss, train_acc, val_loss, val_acc))

# test your model
test_loss, test_acc = test(cnn_model, device, cnn_test_loader)
print('\nTest Loss: {:.6f} \t Test Accuracy: {:.6f}'.format(test_loss, test_acc) )

Train Epoch: 01 	Training Loss: 0.422546 	Training Acc: 0.873033 	Validation Loss: 0.127357 	Validation Acc: 0.956800

Test Loss: 0.058212 	 Test Accuracy: 0.982000


## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Fill out the answers to all questions and submit your file hw6.ipynb to the HW6 assignment on Gradescope. You are free to resubmit as many times as you wish. If the code below throws an error about not being able to generate a PDF, don't worry about it. Your notebook should still be okay.

In [107]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False, run_tests=True)

Running your submission against local test cases...


Your submission received the following results when run against available test cases:

    q1.1 results: All test cases passed!

    q1.2 results: All test cases passed!

    q1.3 results: All test cases passed!

    q1.4 results: All test cases passed!

    q3.1 results: All test cases passed!

    q3.2 results: All test cases passed!

    q3.3 results: All test cases passed!

    q4.1.1 results: All test cases passed!

    q4.1.2 results: All test cases passed!

    q5.1 results: All test cases passed!

    q5.2 results: All test cases passed!

    q5.3.1 results: All test cases passed!

    q5.3.2 results: All test cases passed!

    q5.4 results: All test cases passed!

    q5.4.1 results: All test cases passed!

    q1.5 results: All test cases passed!

    q2 results: All test cases passed!

    q4.2.1 results: All test cases passed!

    q4.2.2 results: All test cases passed!

    q4.2.3 results: All test cases passed!

    q

c:\Users\mapei\Documents\ese577\.venv\Lib\site-packages\otter\check\utils.py:115: RuntimeWarning: [IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
I0000 00:00:1792637510.653247   36860 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792637515.616425   36860 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

  warnings.warn(results.stderr.decode("utf-8"), RuntimeW